In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split

# 读取清洗后的数据
df = pd.read_csv("../data_clean/cleaned_data.csv")

# 定义特征和目标
target = "Churned"
features = [col for col in df.columns if col != target]

X = df[features]
y = df[target]

# 划分训练集和测试集
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("训练集大小:", X_train.shape)
print("测试集大小:", X_test.shape)

训练集大小: (40000, 31)
测试集大小: (10000, 31)


In [4]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, classification_report
from sklearn.preprocessing import StandardScaler

# 只取数值特征
num_cols = X_train.select_dtypes(include="number").columns

X_train_num = X_train[num_cols]
X_test_num = X_test[num_cols]

# 标准化
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_num)
X_test_scaled = scaler.transform(X_test_num)

# 建模
lr = LogisticRegression(max_iter=1000)
lr.fit(X_train_scaled, y_train)

# 预测
y_pred = lr.predict(X_test_scaled)
y_prob = lr.predict_proba(X_test_scaled)[:,1]

# 评估
print("ROC AUC:", roc_auc_score(y_test, y_prob))
print(classification_report(y_test, y_pred))

ROC AUC: 0.7932436404693424
              precision    recall  f1-score   support

           0       0.80      0.92      0.86      7110
           1       0.68      0.45      0.54      2890

    accuracy                           0.78     10000
   macro avg       0.74      0.68      0.70     10000
weighted avg       0.77      0.78      0.76     10000



In [5]:
import numpy as np
from sklearn.metrics import precision_score, recall_score

thresholds = np.arange(0.1, 0.9, 0.05)

for t in thresholds:
    y_pred_adj = (y_prob >= t).astype(int)
    print(f"Threshold: {t:.2f}")
    print("Recall:", recall_score(y_test, y_pred_adj))
    print("Precision:", precision_score(y_test, y_pred_adj))
    print("-"*30)

Threshold: 0.10
Recall: 0.9429065743944637
Precision: 0.35012206090196585
------------------------------
Threshold: 0.15
Recall: 0.9003460207612457
Precision: 0.3905734013809667
------------------------------
Threshold: 0.20
Recall: 0.8432525951557094
Precision: 0.4303372770616281
------------------------------
Threshold: 0.25
Recall: 0.785121107266436
Precision: 0.4758808724832215
------------------------------
Threshold: 0.30
Recall: 0.7179930795847751
Precision: 0.5161691542288557
------------------------------
Threshold: 0.35
Recall: 0.6557093425605537
Precision: 0.5542556303012577
------------------------------
Threshold: 0.40
Recall: 0.5930795847750865
Precision: 0.598463687150838
------------------------------
Threshold: 0.45
Recall: 0.5183391003460207
Precision: 0.6418166238217652
------------------------------
Threshold: 0.50
Recall: 0.44740484429065747
Precision: 0.6841269841269841
------------------------------
Threshold: 0.55
Recall: 0.3726643598615917
Precision: 0.72574123

In [6]:
# 选择最终阈值
best_threshold = 0.35

# 生成最终预测标签
y_pred_final = (y_prob >= best_threshold).astype(int)

from sklearn.metrics import classification_report
print(classification_report(y_test, y_pred_final))

              precision    recall  f1-score   support

           0       0.85      0.79      0.82      7110
           1       0.55      0.66      0.60      2890

    accuracy                           0.75     10000
   macro avg       0.70      0.72      0.71     10000
weighted avg       0.76      0.75      0.75     10000



In [7]:
# 把预测概率加回测试集
test_result = X_test.copy()
test_result["true_churn"] = y_test.values
test_result["churn_probability"] = y_prob

# 分成3档
test_result["risk_level"] = pd.qcut(
    test_result["churn_probability"],
    q=3,
    labels=["Low", "Medium", "High"]
)

test_result.groupby("risk_level")["true_churn"].mean()

C:\Users\lenovo\AppData\Local\Temp\ipykernel_27836\4238155305.py:13: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  test_result.groupby("risk_level")["true_churn"].mean()


risk_level
Low       0.085783
Medium    0.221489
High      0.559688
Name: true_churn, dtype: float64

In [8]:
overall_churn = test_result["true_churn"].mean()
high_churn = test_result[test_result["risk_level"]=="High"]["true_churn"].mean()

print("Overall churn:", overall_churn)
print("High risk churn:", high_churn)
print("Lift:", high_churn / overall_churn)

Overall churn: 0.289
High risk churn: 0.5596880623875224
Lift: 1.9366368940744723
